In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/priyankasingh27/grandstaff/train-00000-of-00001.parquet
/kaggle/input/datasets/priyankasingh27/grandstaff/test-00000-of-00001.parquet
/kaggle/input/datasets/priyankasingh27/grandstaff/val-00000-of-00001.parquet


In [2]:
import io, json, math, os, random, collections, time
import numpy as np, pandas as pd, torch
from PIL import Image

CFG = dict(
    arch="transformer",          # "transformer" | "rnn"
    epochs=20, batch_size=16, lr=3e-4, warmup_steps=500,
    dropout=0.1, d_model=256, layers=4, heads=8,
    max_train_w=1600, max_train_tok=1000,
    val_subset=1000, seed=0,
    out_dir="/kaggle/working",
)
H = 256
dev = "cuda"

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(CFG["seed"])

root = "/kaggle/input/datasets/priyankasingh27/grandstaff"
dfs = {s: pd.read_parquet(f"{root}/{s}-00000-of-00001.parquet") for s in ["train", "val", "test"]}
print({s: len(d) for s, d in dfs.items()})        # 41598 / 4623 / 7661

# ---- lossless tokenizer ----
SEP = {"<s>": " ", "<t>": "\t", "<b>": "\n"}

def tokenize(s):
    s = s.replace(" ", " <s> ").replace("\t", " <t> ").replace("\n", " <b> ").replace("·", " ")
    return s.split()

def detokenize(toks):
    out, prev_sep = [], True
    for t in toks:
        if t in SEP:
            out.append(SEP[t]); prev_sep = True
        else:
            if not prev_sep: out.append("·")
            out.append(t); prev_sep = False
    return "".join(out)

for split, d in dfs.items():
    print(split, "round-trip failures:", sum(detokenize(tokenize(s)) != s for s in d["transcription"]))

{'train': 41598, 'val': 4623, 'test': 7661}
train round-trip failures: 0
val round-trip failures: 0
test round-trip failures: 0


In [3]:
from torch.utils.data import Dataset, DataLoader, Subset, Sampler
from torch.nn.utils.rnn import pad_sequence

# ---- vocab from train only, saved for inference ----
counter = collections.Counter(t for s in dfs["train"]["transcription"] for t in tokenize(s))
SPECIALS = ["<pad>", "<bos>", "<eos>", "<unk>"]
itos = SPECIALS + sorted(counter)
stoi = {t: i for i, t in enumerate(itos)}
PAD, BOS, EOS, UNK = 0, 1, 2, 3
json.dump(itos, open(f"{CFG['out_dir']}/vocab.json", "w"))

def encode(s):
    return [BOS] + [stoi.get(t, UNK) for t in tokenize(s)] + [EOS]

def decode(ids):
    toks = []
    for i in ids:
        if i == EOS: break
        if i not in (PAD, BOS): toks.append(itos[i])
    return detokenize(toks)

class GrandStaff(Dataset):
    def __init__(self, df, augment=None):          # augment hook stays None until Step 7
        self.img_bytes = [d["bytes"] for d in df["image"]]
        self.texts = df["transcription"].tolist()
        self.widths = [Image.open(io.BytesIO(b)).size[0] for b in self.img_bytes]
        self.ntok = [len(tokenize(s)) for s in self.texts]
        self.augment = augment
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, i):
        img = Image.open(io.BytesIO(self.img_bytes[i])).convert("L")
        if self.augment is not None:
            img = self.augment(img)
        a = 1.0 - np.asarray(img, dtype=np.float32) / 255.0      # ink = 1
        if a.shape[0] < H:
            a = np.pad(a, ((0, H - a.shape[0]), (0, 0)))
        return torch.from_numpy(a).unsqueeze(0), torch.tensor(encode(self.texts[i]))

def collate(batch):
    imgs, seqs = zip(*batch)
    widths = torch.tensor([im.shape[-1] for im in imgs])
    x = torch.zeros(len(imgs), 1, H, int(widths.max()))
    for k, im in enumerate(imgs):
        x[k, :, :, : im.shape[-1]] = im
    seqs = pad_sequence(seqs, batch_first=True, padding_value=PAD)
    return {"image": x, "widths": widths, "tgt_in": seqs[:, :-1], "tgt_out": seqs[:, 1:]}

class WidthBucketSampler(Sampler):
    def __init__(self, widths, batch_size, shuffle=True, chunk=50):
        self.w, self.bs, self.shuffle, self.chunk = np.array(widths), batch_size, shuffle, chunk
    def __iter__(self):
        idx = np.random.permutation(len(self.w)) if self.shuffle else np.arange(len(self.w))
        step, batches = self.bs * self.chunk, []
        for s in range(0, len(idx), step):
            c = idx[s:s + step]
            c = c[np.argsort(self.w[c])]
            batches += [c[j:j + self.bs].tolist() for j in range(0, len(c), self.bs)]
        if self.shuffle: random.shuffle(batches)
        return iter(batches)
    def __len__(self):
        return -(-len(self.w) // self.bs)

train_ds, val_ds, test_ds = (GrandStaff(dfs[s]) for s in ("train", "val", "test"))

# training-only filter; val and test are never filtered
keep = [i for i in range(len(train_ds))
        if train_ds.widths[i] <= CFG["max_train_w"] and train_ds.ntok[i] <= CFG["max_train_tok"]]
train_dl = DataLoader(Subset(train_ds, keep), collate_fn=collate, num_workers=2,
                      persistent_workers=True,
                      batch_sampler=WidthBucketSampler([train_ds.widths[i] for i in keep],
                                                       CFG["batch_size"]))

# fixed val subset for model selection
val_idx = sorted(np.random.RandomState(123).choice(len(val_ds), CFG["val_subset"],
                                                   replace=False).tolist())

# checks
print("train kept:", len(keep), "/", len(train_ds))              # expect 40684 / 41598
print("vocab incl. specials:", len(itos))                        # expect 188
print("id round-trip failures:", sum(decode(encode(s)) != s for s in train_ds.texts))
b = next(iter(train_dl))
print({k: tuple(v.shape) for k, v in b.items()}, b["image"].min().item(), b["image"].max().item())

train kept: 40684 / 41598
vocab incl. specials: 188
id round-trip failures: 0
{'image': (16, 1, 256, 1408), 'widths': (16,), 'tgt_in': (16, 976), 'tgt_out': (16, 976)} 0.0 1.0


In [4]:
bad = sum(decode(encode(s)) != s for s in test_ds.texts)
print("test id round-trip failures:", bad)     # expect ~5036 of 7661, all from n

test id round-trip failures: 5036


In [5]:
import torchvision
import torch.nn as nn, torch.nn.functional as F

def pos2d(d, h, w, device):
    q = d // 4
    freq = torch.exp(torch.arange(q, device=device) * (-math.log(10000.0) / q))
    ys = torch.arange(h, device=device)[:, None] * freq[None]
    xs = torch.arange(w, device=device)[:, None] * freq[None]
    pe_y = torch.cat([ys.sin(), ys.cos()], 1)
    pe_x = torch.cat([xs.sin(), xs.cos()], 1)
    return torch.cat([pe_y[:, None].expand(h, w, d // 2),
                      pe_x[None].expand(h, w, d // 2)], 2)           # (h, w, d)

def pos1d(T, d, device):
    pos = torch.arange(T, device=device)[:, None]
    freq = torch.exp(torch.arange(0, d, 2, device=device) * (-math.log(10000.0) / d))
    pe = torch.zeros(T, d, device=device)
    pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * freq), torch.cos(pos * freq)
    return pe

class OMRModel(nn.Module):
    STRIDE = 16
    def __init__(self, vocab, d=256, layers=4, heads=8, ff=1024, drop=0.1):
        super().__init__()
        r = torchvision.models.resnet34(weights=None)
        r.conv1 = nn.Conv2d(1, 64, 7, 2, 3, bias=False)
        self.backbone = nn.Sequential(r.conv1, r.bn1, r.relu, r.maxpool,
                                      r.layer1, r.layer2, r.layer3)
        self.proj = nn.Conv2d(256, d, 1)
        self.embed = nn.Embedding(vocab, d, padding_idx=PAD)
        self.drop = nn.Dropout(drop)
        layer = nn.TransformerDecoderLayer(d, heads, ff, drop, batch_first=True, norm_first=True)
        self.decoder = nn.TransformerDecoder(layer, layers, norm=nn.LayerNorm(d))
        self.out = nn.Linear(d, vocab)

    def encode_img(self, img, widths):
        f = self.proj(self.backbone(img))                            # (B, d, h, w)
        B, d, h, w = f.shape
        f = f + pos2d(d, h, w, f.device).permute(2, 0, 1)[None].to(f.dtype)
        mem = f.flatten(2).transpose(1, 2)                           # (B, h*w, d)
        valid = torch.ceil(widths.to(f.device) / self.STRIDE).long()
        col_pad = torch.arange(w, device=f.device)[None] >= valid[:, None]
        mem_pad = col_pad[:, None, :].expand(B, h, w).reshape(B, h * w)
        return mem, mem_pad

    def decode_tokens(self, tgt_in, mem, mem_pad):
        T = tgt_in.size(1)
        x = self.embed(tgt_in) + pos1d(T, self.embed.embedding_dim, tgt_in.device)[None]
        x = self.drop(x)
        causal = torch.triu(torch.ones(T, T, dtype=torch.bool, device=x.device), 1)
        return self.out(self.decoder(x, mem, tgt_mask=causal, memory_key_padding_mask=mem_pad))

    def forward(self, img, widths, tgt_in):
        mem, mem_pad = self.encode_img(img, widths)
        return self.decode_tokens(tgt_in, mem, mem_pad)

model = OMRModel(len(itos), d=CFG["d_model"], layers=CFG["layers"],
                 heads=CFG["heads"], drop=CFG["dropout"]).to(dev)
print("params (M):", sum(p.numel() for p in model.parameters()) / 1e6)   # ~13.0

# checks: real batch, and a 1700-token sequence (no learned position limit)
b = next(iter(train_dl))
with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
    print(model(b["image"].to(dev), b["widths"], b["tgt_in"].to(dev)).shape)   # (16, 976, 188)
    mem, mp = model.encode_img(b["image"][:2].to(dev), b["widths"][:2])
    print(model.decode_tokens(torch.zeros(2, 1700, dtype=torch.long, device=dev), mem, mp).shape)

params (M): 12.54054
torch.Size([16, 673, 188])
torch.Size([2, 1700, 188])


In [6]:
train_dl = DataLoader(Subset(train_ds, keep), collate_fn=collate, num_workers=2,
                      batch_sampler=WidthBucketSampler([train_ds.widths[i] for i in keep],
                                                       CFG["batch_size"]))

b = next(iter(train_dl))
with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
    print(model(b["image"].to(dev), b["widths"], b["tgt_in"].to(dev)).shape)   # (16, T, 188)
    mem, mp = model.encode_img(b["image"][:2].to(dev), b["widths"][:2])
    print(model.decode_tokens(torch.zeros(2, 1700, dtype=torch.long, device=dev), mem, mp).shape)

torch.Size([16, 498, 188])
torch.Size([2, 1700, 188])


In [7]:
import torch.nn as nn, torch.nn.functional as F

@torch.no_grad()
def greedy(model, batch, max_len=1800):
    model.eval()
    with torch.autocast("cuda", dtype=torch.float16):
        mem, mem_pad = model.encode_img(batch["image"].to(dev), batch["widths"])
        ys = torch.full((mem.size(0), 1), BOS, device=dev)
        done = torch.zeros(mem.size(0), dtype=torch.bool, device=dev)
        for _ in range(max_len):
            nxt = model.decode_tokens(ys, mem, mem_pad)[:, -1].argmax(-1)
            nxt = torch.where(done, torch.full_like(nxt, PAD), nxt)
            ys = torch.cat([ys, nxt[:, None]], 1)
            done |= nxt == EOS
            if done.all(): break
    return ys[:, 1:].tolist()

def batch_loss(model, img, widths, b):
    with torch.autocast("cuda", dtype=torch.float16):
        logits = model(img.to(dev), widths, b["tgt_in"].to(dev))
    return F.cross_entropy(logits.float().reshape(-1, logits.size(-1)),
                           b["tgt_out"].to(dev).reshape(-1), ignore_index=PAD)

# ---- overfit 20 samples, no dropout ----
gate = OMRModel(len(itos), d=CFG["d_model"], layers=CFG["layers"],
                heads=CFG["heads"], drop=0.0).to(dev)
opt = torch.optim.AdamW(gate.parameters(), lr=3e-4, weight_decay=0.01)
scaler = torch.amp.GradScaler("cuda")
ov = DataLoader(Subset(train_ds, range(20)), batch_size=10, shuffle=True, collate_fn=collate)

t0 = time.time()
for epoch in range(400):
    gate.train(); tot = 0
    for b in ov:
        loss = batch_loss(gate, b["image"], b["widths"], b)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt); nn.utils.clip_grad_norm_(gate.parameters(), 1.0)
        scaler.step(opt); scaler.update(); tot += loss.item()
    tot /= len(ov)
    if epoch % 25 == 0: print(epoch, round(tot, 4), f"{time.time()-t0:.0f}s")
    if tot < 0.005: print("stopped at epoch", epoch, "loss", round(tot, 4)); break

# ---- exact match on all 20 ----
gate.eval(); ok = 0
for s in (0, 10):
    b = collate([train_ds[i] for i in range(s, s + 10)])
    for k, p in enumerate(greedy(gate, b)):
        ref, hyp = tokenize(train_ds.texts[s + k]), tokenize(decode(p))
        first = next((i for i, (r, h) in enumerate(zip(ref, hyp)) if r != h), None)
        if first is None and len(ref) != len(hyp): first = min(len(ref), len(hyp))
        ok += first is None
        if first is not None: print(s + k, "first mismatch at token", first, "of", len(ref))
print("exact match:", ok, "/ 20")

# ---- does the decoder use the image? ----
b = collate([train_ds[i] for i in range(10)])
with torch.no_grad():
    real = batch_loss(gate, b["image"], b["widths"], b).item()
    zero = batch_loss(gate, torch.zeros_like(b["image"]), b["widths"], b).item()
    swap = batch_loss(gate, b["image"].roll(1, 0), b["widths"].roll(1, 0), b).item()
print(f"loss real {real:.4f} | zeroed {zero:.4f} | swapped {swap:.4f}")

del gate, opt; torch.cuda.empty_cache()

0 4.9635 3s
25 1.7428 16s
50 1.2183 26s
75 0.8142 37s
100 0.5015 48s
125 0.2518 59s
150 0.1057 70s
175 0.0716 82s
200 0.0353 94s
225 0.0169 106s
250 0.0191 118s
275 0.0139 129s
stopped at epoch 284 loss 0.0046
exact match: 20 / 20
loss real 0.0045 | zeroed 2.9859 | swapped 3.4547


In [8]:
CFG.update(eval_every=2, ser_n=300, lr_floor=0.05, grad_clip=1.0, resume=True)

# ---------- metrics ----------
try:
    import editdistance
    edit = editdistance.eval
except Exception:
    def edit(a, b):
        prev = list(range(len(b) + 1))
        for i, x in enumerate(a, 1):
            cur = [i]
            for j, y in enumerate(b, 1):
                cur.append(min(prev[j] + 1, cur[j-1] + 1, prev[j-1] + (x != y)))
            prev = cur
        return prev[-1]

def ids_to_tokens(ids):
    out = []
    for i in ids:
        if i == EOS: break
        if i not in (PAD, BOS): out.append(itos[i])
    return out

@torch.no_grad()
def evaluate_ser(model, ds, idx, bs=16, max_len=1800, drop_n=False):
    """token edit distance / reference length; returns (mean per-sample, corpus-level), in %"""
    idx = sorted(idx, key=lambda i: ds.widths[i])
    per, errs, tot = [], 0, 0
    for s in range(0, len(idx), bs):
        chunk = idx[s:s + bs]
        batch = collate([ds[i] for i in chunk])
        refs = [tokenize(ds.texts[i]) for i in chunk]
        if drop_n: refs = [[t for t in r if t != "n"] for r in refs]
        for p, r in zip(greedy(model, batch, max_len=max_len), refs):
            e = edit(ids_to_tokens(p), r)
            per.append(e / len(r)); errs += e; tot += len(r)
    return 100 * float(np.mean(per)), 100 * errs / tot

@torch.no_grad()
def val_loss(model, ds, idx):
    dl = DataLoader(Subset(ds, idx), collate_fn=collate, num_workers=2,
                    batch_sampler=WidthBucketSampler([ds.widths[i] for i in idx],
                                                     CFG["batch_size"], shuffle=False))
    model.eval(); tot, n = 0.0, 0
    for b in dl:
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(b["image"].to(dev), b["widths"], b["tgt_in"].to(dev))
        tgt = b["tgt_out"].to(dev)
        tot += F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), tgt.reshape(-1),
                               ignore_index=PAD, reduction="sum").item()
        n += (tgt != PAD).sum().item()
    return tot / n

# ---------- optimizer / schedule ----------
def make_opt(model):
    decay, no_decay = [], []
    for n, p in model.named_parameters():
        (no_decay if p.ndim < 2 or "embed" in n else decay).append(p)
    return torch.optim.AdamW([{"params": decay, "weight_decay": 0.01},
                              {"params": no_decay, "weight_decay": 0.0}], lr=CFG["lr"])

total_steps = CFG["epochs"] * len(train_dl)
def lr_lambda(step):
    if step < CFG["warmup_steps"]:
        return (step + 1) / CFG["warmup_steps"]
    t = (step - CFG["warmup_steps"]) / max(1, total_steps - CFG["warmup_steps"])
    return CFG["lr_floor"] + (1 - CFG["lr_floor"]) * 0.5 * (1 + math.cos(math.pi * t))

# ---------- build / resume ----------
seed_all(CFG["seed"])
model = OMRModel(len(itos), d=CFG["d_model"], layers=CFG["layers"],
                 heads=CFG["heads"], drop=CFG["dropout"]).to(dev)
opt = make_opt(model)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)
scaler = torch.amp.GradScaler("cuda")
last_p, best_p = f"{CFG['out_dir']}/last.pt", f"{CFG['out_dir']}/best.pt"
start_ep, best, hist = 0, 1e9, []
if CFG["resume"] and os.path.exists(last_p):
    ck = torch.load(last_p, map_location=dev)
    model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
    sched.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"])
    start_ep, best, hist = ck["epoch"] + 1, ck["best"], ck["hist"]
    print("resumed from epoch", ck["epoch"])

ser_idx = val_idx[::max(1, len(val_idx) // CFG["ser_n"])][:CFG["ser_n"]]
print("params (M):", sum(p.numel() for p in model.parameters()) / 1e6, "| steps/epoch:", len(train_dl))

# ---------- train ----------
for ep in range(start_ep, CFG["epochs"]):
    model.train(); run, t0 = 0.0, time.time()
    for step, b in enumerate(train_dl):
        loss = batch_loss(model, b["image"], b["widths"], b)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
        scaler.step(opt); scaler.update(); sched.step()
        run += loss.item()
        if step % 400 == 0:
            print(f"ep {ep} step {step}/{len(train_dl)} loss {run/(step+1):.4f} "
                  f"lr {sched.get_last_lr()[0]:.2e} {(time.time()-t0)/(step+1):.2f}s/step")
    tr = run / len(train_dl)
    vl = val_loss(model, val_ds, val_idx)
    rec = dict(epoch=ep, train=tr, val_loss=vl)
    msg = f"EPOCH {ep} | train {tr:.4f} | val loss {vl:.4f}"
    if (ep + 1) % CFG["eval_every"] == 0 or ep == CFG["epochs"] - 1:
        ser, ser_c = evaluate_ser(model, val_ds, ser_idx)
        rec.update(ser=ser, ser_corpus=ser_c); msg += f" | val SER {ser:.2f} (corpus {ser_c:.2f})"
        if ser < best:
            best = ser; torch.save({"model": model.state_dict(), "epoch": ep, "ser": ser}, best_p)
            msg += " *best*"
    hist.append(rec)
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                "scaler": scaler.state_dict(), "epoch": ep, "best": best, "hist": hist}, last_p)
    json.dump(hist, open(f"{CFG['out_dir']}/history.json", "w"))
    print(msg, f"| {(time.time()-t0)/60:.1f} min")

params (M): 12.54054 | steps/epoch: 2543
ep 0 step 0/2543 loss 5.3231 lr 1.20e-06 1.09s/step
ep 0 step 400/2543 loss 2.4846 lr 2.41e-04 0.60s/step
ep 0 step 800/2543 loss 2.0231 lr 3.00e-04 0.52s/step
ep 0 step 1200/2543 loss 1.7270 lr 3.00e-04 0.47s/step
ep 0 step 1600/2543 loss 1.5243 lr 3.00e-04 0.43s/step
ep 0 step 2000/2543 loss 1.3713 lr 2.99e-04 0.41s/step
ep 0 step 2400/2543 loss 1.2489 lr 2.99e-04 0.39s/step
EPOCH 0 | train 1.2110 | val loss 0.5019 | 16.2 min
ep 1 step 0/2543 loss 0.5995 lr 2.99e-04 0.54s/step
ep 1 step 400/2543 loss 0.5081 lr 2.98e-04 0.26s/step
ep 1 step 800/2543 loss 0.4714 lr 2.98e-04 0.26s/step
ep 1 step 1200/2543 loss 0.4390 lr 2.97e-04 0.26s/step
ep 1 step 1600/2543 loss 0.4112 lr 2.96e-04 0.25s/step
ep 1 step 2000/2543 loss 0.3848 lr 2.95e-04 0.25s/step
ep 1 step 2400/2543 loss 0.3617 lr 2.95e-04 0.25s/step
EPOCH 1 | train 0.3552 | val loss 0.2010 | val SER 27.60 (corpus 36.86) *best* | 18.6 min
ep 2 step 0/2543 loss 0.2127 lr 2.94e-04 0.53s/step
ep 2 